#FEATURE ENGINEERING

In [1]:
import pandas as pd
import numpy as np

retail_df = pd.read_csv("../../02_Cleaned_Data/online_retail_cleaned.csv")

retail_df.shape

(779425, 8)

In [2]:
retail_df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [3]:
retail_df["Revenue"] = retail_df["Quantity"] * retail_df["Price"]

In [4]:
retail_df[["Quantity", "Price", "Revenue"]].head()

,Quantity,Price,Revenue
0,12,6.95,83.4
1,12,6.75,81.0
2,12,6.75,81.0
3,48,2.10,100.8
4,24,1.25,30.0


In [5]:
retail_df["Revenue"].describe()

count    779425.000000
mean         22.291823
std         227.427075
min           0.001000
25%           4.950000
50%          12.480000
75%          19.800000
max      168469.600000
Name: Revenue, dtype: float64

In [6]:
retail_df["InvoiceDate"]= pd.to_datetime(retail_df["InvoiceDate"])
retail_df["InvoiceDate"].head()

0   2009-12-01 07:45:00
1   2009-12-01 07:45:00
2   2009-12-01 07:45:00
3   2009-12-01 07:45:00
4   2009-12-01 07:45:00
Name: InvoiceDate, dtype: datetime64[us]

In [7]:
retail_df["Year"] = retail_df["InvoiceDate"].dt.year
retail_df["Month"] = retail_df["InvoiceDate"].dt.month
retail_df["Day"] = retail_df["InvoiceDate"].dt.day
retail_df["Hour"] = retail_df["InvoiceDate"].dt.hour
retail_df["Weekday"] = retail_df["InvoiceDate"].dt.day_name()

retail_df[
    ["InvoiceDate","Year","Month","Day","Hour","Weekday"]
].head()

,InvoiceDate,Year,Month,Day,Hour,Weekday
0,2009-12-01 07:45:00,2009,12,1,7,Tuesday
1,2009-12-01 07:45:00,2009,12,1,7,Tuesday
2,2009-12-01 07:45:00,2009,12,1,7,Tuesday
3,2009-12-01 07:45:00,2009,12,1,7,Tuesday
4,2009-12-01 07:45:00,2009,12,1,7,Tuesday


In [8]:
retail_df["Customer ID"] = retail_df["Customer ID"].astype(int)

retail_df["Customer ID"].head()

0    13085
1    13085
2    13085
3    13085
4    13085
Name: Customer ID, dtype: int64

In [9]:
retail_df["Customer ID"].nunique()

5878

In [10]:
customer_revenue= retail_df.groupby("Customer ID")["Revenue"].sum().reset_index()

customer_revenue.head()

,Customer ID,Revenue
0,12346,77556.46
1,12347,4921.53
2,12348,2019.40
3,12349,4428.69
4,12350,334.40


In [11]:
customer_frequency = retail_df.groupby("Customer ID")["Invoice"].nunique().reset_index()

customer_frequency.columns = ["Customer ID", "Purchase_Frequency"]

customer_frequency.head()

,Customer ID,Purchase_Frequency
0,12346,12
1,12347,8
2,12348,5
3,12349,4
4,12350,1


In [12]:
customer_frequency.shape

(5878, 2)

In [13]:
customer_monetary = retail_df.groupby("Customer ID")["Revenue"].sum().reset_index()

customer_monetary.columns = ["Customer ID", "Monetary_Value"]

customer_monetary.head()

,Customer ID,Monetary_Value
0,12346,77556.46
1,12347,4921.53
2,12348,2019.40
3,12349,4428.69
4,12350,334.40


In [14]:
customer_monetary.shape

(5878, 2)

In [15]:
snapshot_date = retail_df["InvoiceDate"].max()

snapshot_date

Timestamp('2011-12-09 12:50:00')

In [16]:
customer_recency = retail_df.groupby("Customer ID")["InvoiceDate"].max().reset_index()

customer_recency["Recency"] = (
    snapshot_date - customer_recency["InvoiceDate"]
).dt.days

customer_recency.head()

,Customer ID,InvoiceDate,Recency
0,12346,2011-01-18 10:01:00,325
1,12347,2011-12-07 15:52:00,1
2,12348,2011-09-25 13:13:00,74
3,12349,2011-11-21 09:51:00,18
4,12350,2011-02-02 16:01:00,309


In [17]:
customer_recency.shape

(5878, 3)

In [18]:
rfm_df = customer_recency.merge(
    customer_frequency,
    on="Customer ID"
)

rfm_df = rfm_df.merge(
    customer_monetary,
    on="Customer ID"
)

rfm_df.head()

,Customer ID,InvoiceDate,Recency,Purchase_Frequency,Monetary_Value
0,12346,2011-01-18 10:01:00,325,12,77556.46
1,12347,2011-12-07 15:52:00,1,8,4921.53
2,12348,2011-09-25 13:13:00,74,5,2019.40
3,12349,2011-11-21 09:51:00,18,4,4428.69
4,12350,2011-02-02 16:01:00,309,1,334.40


In [19]:
rfm_df.shape

(5878, 5)

In [20]:
rfm_df[
    ["Recency", "Purchase_Frequency", "Monetary_Value"]
].describe()

,Recency,Purchase_Frequency,Monetary_Value
count,5878.000000,5878.000000,5878.000000
mean,200.331916,6.289384,2955.904095
std,209.338707,13.009406,14440.852688
min,0.000000,1.000000,2.950000
25%,25.000000,1.000000,342.280000
50%,95.000000,3.000000,867.740000
75%,379.000000,7.000000,2248.305000
max,738.000000,398.000000,580987.040000


In [21]:
rfm_df["R_Score"] = pd.qcut(
    rfm_df["Recency"],
    5,
    labels=[5,4,3,2,1]
)

rfm_df["F_Score"] = pd.qcut(
    rfm_df["Purchase_Frequency"].rank(method="first"),
    5,
    labels=[1,2,3,4,5]
)

rfm_df["M_Score"] = pd.qcut(
    rfm_df["Monetary_Value"].rank(method="first"),
    5,
    labels=[1,2,3,4,5]
)

rfm_df[
    ["Recency","Purchase_Frequency","Monetary_Value",
     "R_Score","F_Score","M_Score"]
].head()

,Recency,Purchase_Frequency,Monetary_Value,R_Score,F_Score,M_Score
0,325,12,77556.46,2,5,5
1,1,8,4921.53,5,4,5
2,74,5,2019.40,3,4,4
3,18,4,4428.69,5,3,5
4,309,1,334.40,2,1,2


In [22]:
rfm_df["RFM_Score"] = (
    rfm_df["R_Score"].astype(str) +
    rfm_df["F_Score"].astype(str) +
    rfm_df["M_Score"].astype(str)
)

rfm_df[
    ["Customer ID","R_Score","F_Score","M_Score","RFM_Score"]
].head()

,Customer ID,R_Score,F_Score,M_Score,RFM_Score
0,12346,2,5,5,255
1,12347,5,4,5,545
2,12348,3,4,4,344
3,12349,5,3,5,535
4,12350,2,1,2,212


# Feature Engineering Summary

## Objective
Create business-ready features for analytics and machine learning.

## Key Findings
- Revenue feature was created.
- Date-based features were generated.
- Customer-level metrics became easier to analyze.
- Dataset became suitable for advanced analytics.

## Recommendations
- Use engineered features in customer segmentation.
- Reuse feature pipeline for future datasets.
- Continuously evaluate feature importance.
- Create additional behavioral features if required.

# ML Version 2 - Advanced Feature Engineering

## Objective

Create business-driven engineered features to improve customer value prediction performance beyond raw RFM metrics.

## Business Goal

Evaluate whether engineered behavioral features can improve classification accuracy compared to Version 1 models.

## Expected Output

A new machine learning dataset:

customer_clv_v2.csv

This dataset will be used for:

- Logistic Regression
- Random Forest
- XGBoost
- Deep Learning

## Hypothesis

Customers with similar RFM scores may still behave differently.

Additional behavioral features may help machine learning models better distinguish High Value, Medium Value, and Low Value customers.

In [26]:
rfm_df.columns.tolist()

['Customer ID',
 'InvoiceDate',
 'Recency',
 'Purchase_Frequency',
 'Monetary_Value',
 'R_Score',
 'F_Score',
 'M_Score',
 'RFM_Score']

In [27]:
rfm_df.head()

,Customer ID,InvoiceDate,Recency,Purchase_Frequency,Monetary_Value,R_Score,F_Score,M_Score,RFM_Score
0,12346,2011-01-18 10:01:00,325,12,77556.46,2,5,5,255
1,12347,2011-12-07 15:52:00,1,8,4921.53,5,4,5,545
2,12348,2011-09-25 13:13:00,74,5,2019.40,3,4,4,344
3,12349,2011-11-21 09:51:00,18,4,4428.69,5,3,5,535
4,12350,2011-02-02 16:01:00,309,1,334.40,2,1,2,212


## Feature 1: Customer Tenure

### Objective

Create a customer tenure feature to measure how long a customer has been active in the business.

### Business Value

Customers with longer tenure often exhibit different purchasing behavior and customer lifetime value patterns compared to newer customers.

In [28]:
snapshot_date


Timestamp('2011-12-09 12:50:00')

In [29]:
rfm_df["Customer_Tenure"] = (
    snapshot_date - rfm_df["InvoiceDate"]
).dt.days

In [30]:
rfm_df[
    ["Customer ID", "InvoiceDate", "Customer_Tenure"]
].head()

,Customer ID,InvoiceDate,Customer_Tenure
0,12346,2011-01-18 10:01:00,325
1,12347,2011-12-07 15:52:00,1
2,12348,2011-09-25 13:13:00,74
3,12349,2011-11-21 09:51:00,18
4,12350,2011-02-02 16:01:00,309


## Feature 2: Average Order Value

### Objective

Create a feature representing the average revenue generated per purchase.

### Business Value

Customers with similar revenue may have very different purchasing patterns.

Average Order Value helps distinguish:
- High-spending occasional buyers
- Low-spending frequent buyers

This feature may improve customer value prediction performance.

In [31]:
rfm_df["Average_Order_Value"] = (
    rfm_df["Monetary_Value"] /
    rfm_df["Purchase_Frequency"]
)

In [32]:
rfm_df[
    [
        "Monetary_Value",
        "Purchase_Frequency",
        "Average_Order_Value"
    ]
].head()

,Monetary_Value,Purchase_Frequency,Average_Order_Value
0,77556.46,12,6463.038333
1,4921.53,8,615.191250
2,2019.40,5,403.880000
3,4428.69,4,1107.172500
4,334.40,1,334.400000


## Feature 3: Purchase Velocity

### Objective

Measure how frequently customers purchase relative to their relationship duration.

### Business Value

Two customers may have the same purchase frequency but very different tenure lengths.

Purchase Velocity helps identify:

- Highly active customers
- Fast-growing customers
- Consistently engaged customers

This feature may improve customer segmentation and value prediction.

In [33]:
rfm_df["Purchase_Velocity"] = (
    rfm_df["Purchase_Frequency"] /
    (rfm_df["Customer_Tenure"] + 1)
)

In [34]:
rfm_df[
    [
        "Purchase_Frequency",
        "Customer_Tenure",
        "Purchase_Velocity"
    ]
].head()

,Purchase_Frequency,Customer_Tenure,Purchase_Velocity
0,12,325,0.036810
1,8,1,4.000000
2,5,74,0.066667
3,4,18,0.210526
4,1,309,0.003226


## Feature 4: Customer Lifetime Daily Value

### Objective

Measure average revenue generated per day during the customer relationship.

### Business Value

Customers with similar total revenue may generate value at very different rates.

Customer Lifetime Daily Value helps identify:

- Fast value-generating customers
- Premium customers
- High-growth customers

This feature may improve customer value prediction performance.

In [35]:
rfm_df["Daily_Value"] = (
    rfm_df["Monetary_Value"] /
    (rfm_df["Customer_Tenure"] + 1)
)

In [36]:
rfm_df[
    [
        "Monetary_Value",
        "Customer_Tenure",
        "Daily_Value"
    ]
].head()

,Monetary_Value,Customer_Tenure,Daily_Value
0,77556.46,325,237.903252
1,4921.53,1,2460.765000
2,2019.40,74,26.925333
3,4428.69,18,233.088947
4,334.40,309,1.078710


## Feature 5: Customer Engagement Score

### Objective

Create a combined engagement indicator using purchase frequency and customer tenure.

### Business Value

Highly engaged customers tend to purchase frequently and remain active over longer periods.

This feature helps identify:

- Loyal customers
- Repeat buyers
- High-retention customers

The feature may improve machine learning performance by capturing customer engagement behaviour.

In [37]:
rfm_df["Engagement_Score"] = (
    rfm_df["Purchase_Frequency"] *
    rfm_df["Customer_Tenure"]
)

In [38]:
rfm_df[
    [
        "Purchase_Frequency",
        "Customer_Tenure",
        "Engagement_Score"
    ]
].head()

,Purchase_Frequency,Customer_Tenure,Engagement_Score
0,12,325,3900
1,8,1,8
2,5,74,370
3,4,18,72
4,1,309,309


In [39]:
rfm_df.columns.tolist()

['Customer ID',
 'InvoiceDate',
 'Recency',
 'Purchase_Frequency',
 'Monetary_Value',
 'R_Score',
 'F_Score',
 'M_Score',
 'RFM_Score',
 'Customer_Tenure',
 'Average_Order_Value',
 'Purchase_Velocity',
 'Daily_Value',
 'Engagement_Score']

In [40]:
rfm_df.shape

(5878, 14)

In [43]:
customer_clv = pd.read_csv(
    "../../02_Cleaned_Data/customer_clv.csv"
)

customer_clv.head()

,Customer ID,Total_Revenue,Purchase_Frequency,CLV_Segment
0,12346,77556.46,12,High Value
1,12347,4921.53,8,Medium Value
2,12348,2019.40,5,Low Value
3,12349,4428.69,4,Medium Value
4,12350,334.40,1,Low Value


In [44]:
customer_clv.columns.tolist()

['Customer ID', 'Total_Revenue', 'Purchase_Frequency', 'CLV_Segment']

In [45]:
rfm_v2 = pd.merge(
    rfm_df,
    customer_clv[["Customer ID", "CLV_Segment"]],
    on="Customer ID",
    how="left"
)

rfm_v2.shape

(5878, 15)

In [46]:
rfm_v2.head()

,Customer ID,InvoiceDate,Recency,Purchase_Frequency,Monetary_Value,R_Score,F_Score,M_Score,RFM_Score,Customer_Tenure,Average_Order_Value,Purchase_Velocity,Daily_Value,Engagement_Score,CLV_Segment
0,12346,2011-01-18 10:01:00,325,12,77556.46,2,5,5,255,325,6463.038333,0.036810,237.903252,3900,High Value
1,12347,2011-12-07 15:52:00,1,8,4921.53,5,4,5,545,1,615.191250,4.000000,2460.765000,8,Medium Value
2,12348,2011-09-25 13:13:00,74,5,2019.40,3,4,4,344,74,403.880000,0.066667,26.925333,370,Low Value
3,12349,2011-11-21 09:51:00,18,4,4428.69,5,3,5,535,18,1107.172500,0.210526,233.088947,72,Medium Value
4,12350,2011-02-02 16:01:00,309,1,334.40,2,1,2,212,309,334.400000,0.003226,1.078710,309,Low Value


In [47]:
rfm_v2["CLV_Segment"].value_counts()

CLV_Segment
Low Value       4408
Medium Value     882
High Value       588
Name: count, dtype: int64

In [48]:
rfm_v2.to_csv(
    "../../02_Cleaned_Data/customer_clv_ml_v2.csv",
    index=False
)

In [49]:
pd.read_csv(
    "../../02_Cleaned_Data/customer_clv_ml_v2.csv"
).head()

,Customer ID,InvoiceDate,Recency,Purchase_Frequency,Monetary_Value,R_Score,F_Score,M_Score,RFM_Score,Customer_Tenure,Average_Order_Value,Purchase_Velocity,Daily_Value,Engagement_Score,CLV_Segment
0,12346,2011-01-18 10:01:00,325,12,77556.46,2,5,5,255,325,6463.038333,0.036810,237.903252,3900,High Value
1,12347,2011-12-07 15:52:00,1,8,4921.53,5,4,5,545,1,615.191250,4.000000,2460.765000,8,Medium Value
2,12348,2011-09-25 13:13:00,74,5,2019.40,3,4,4,344,74,403.880000,0.066667,26.925333,370,Low Value
3,12349,2011-11-21 09:51:00,18,4,4428.69,5,3,5,535,18,1107.172500,0.210526,233.088947,72,Medium Value
4,12350,2011-02-02 16:01:00,309,1,334.40,2,1,2,212,309,334.400000,0.003226,1.078710,309,Low Value


# Final Conclusion - ML Version 2 Feature Engineering

## Objective

Enhance traditional RFM analysis by creating advanced behavioral features that better capture customer purchasing patterns, engagement levels, and value generation behavior.

## Features Created

### Traditional RFM Features

- Recency
- Purchase_Frequency
- Monetary_Value
- R_Score
- F_Score
- M_Score
- RFM_Score

### Advanced Behavioral Features

- Customer_Tenure
- Average_Order_Value
- Purchase_Velocity
- Daily_Value
- Engagement_Score

## Key Findings

- Customer tenure successfully captured customer lifecycle duration.
- Average Order Value highlighted spending quality beyond purchase frequency.
- Purchase Velocity identified customers with faster purchasing behavior.
- Daily Value measured revenue generation efficiency over time.
- Engagement Score quantified long-term customer activity levels.
- Dataset richness increased significantly beyond traditional RFM metrics.

## Business Impact

- Improved customer profiling capabilities.
- Better identification of high-value customers.
- Enhanced customer segmentation potential.
- Stronger foundation for machine learning models.
- Improved targeting for retention and marketing campaigns.

## Recommendations

- Use engineered behavioral features in predictive models.
- Track customer engagement and purchase velocity regularly.
- Prioritize retention efforts for highly engaged customers.
- Combine RFM and behavioral metrics for customer intelligence initiatives.
- Continuously evaluate feature importance and model performance.

## Deliverable Summary

| Component | Status |
|------------|---------|
| RFM Features | Completed |
| Customer Tenure | Completed |
| Average Order Value | Completed |
| Purchase Velocity | Completed |
| Daily Value | Completed |
| Engagement Score | Completed |
| CLV Integration | Completed |
| Version 2 Dataset Export | Completed |

## Final Dataset

`customer_clv_ml_v2.csv`

## Final Dataset Shape

**5,878 Rows × 15 Columns**

## Next Phase

Advanced Machine Learning Modeling using engineered behavioral features to evaluate performance improvements over the Version 1 baseline models.